In [16]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
print("libraries imported successfully")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")
print(f"json version: {json.__version__}")
print(f"Loaded at:{datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}")


libraries imported successfully
pandas version: 2.2.2
requests version: 2.32.4
json version: 2.0.9
Loaded at:2026-05-27  14:52:06


In [17]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(f"rows:{raw_df.shape[0]}")
print(f"columns:{raw_df.shape[1]}")
print(f"column names: {raw_df.columns.tolist()}")
print("\n first 5 rows of a dataset")
raw_df.head()

rows:30
columns:9
column names: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

 first 5 rows of a dataset


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [18]:
print("="*55)
print("Data quality diagnosis report")
print("="*55)
print("\n[1] MISSING VALUE per column:")
print(raw_df.isnull().sum())
print(f"\n[2] DUPLICATED ROWS: {raw_df.duplicated().sum()}")
print("\n[3] DATATYPES")
print(raw_df.dtypes)
print("\n[4] UNIQUE CATEGORY:",raw_df['category'].dropna().unique().tolist())
print("\n[5] SAMPLE ORDER DATES:",raw_df['order_date'].unique()[:8].tolist())
print("\n[6] SAMPLE NAMES:",raw_df['customer_name'].dropna().unique()[:8].tolist())

Data quality diagnosis report

[1] MISSING VALUE per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATED ROWS: 0

[3] DATATYPES
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORY: ['Electronics', 'Accessories']

[5] SAMPLE ORDER DATES: ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']

[6] SAMPLE NAMES: ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [19]:
df=raw_df.copy()
print(f"working copy created with shape: {df.shape}")
print("raw_df is untouched")

working copy created with shape: (30, 9)
raw_df is untouched


In [20]:
ai=raw_df.copy()
print(f"working copy created with shape: {ai.shape}")
print("raw_df is untouched")

working copy created with shape: (30, 9)
raw_df is untouched


In [21]:
print(f"total before missing fix:{ai.isnull().sum().sum()}")
ai['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty=ai['quantity'].median()
print(f"median quantity: {median_qty}")
ai['quantity'].fillna(median_qty,inplace=True)
print(f"filled missing quantity with median:{median_qty}")
ai['category'].fillna('Unknown category',inplace=True)
print(f"filled missing category with 'Unknown category'")

print(f"total after missing fix:{ai.isnull().sum().sum()}")


total before missing fix:7
median quantity: 2.0
filled missing quantity with median:2.0
filled missing category with 'Unknown category'
total after missing fix:1


In [22]:
print("total before missing fix:",ai.isnull().sum().sum())
ai['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty=ai['quantity'].median()
print(f"median quantity: {median_qty}")
ai['quantity'].fillna(median_qty,inplace=True)
print(f"filled missing quantity with median:{median_qty}")
ai['category'].fillna('Unknown category',inplace=True)
print(f"filled missing category with 'Unknown category'")
ai['product'].fillna('Unknown product',inplace=True)
print(f"total after missing fix:{ai.isnull().sum().sum()}")

total before missing fix: 1
median quantity: 2.0
filled missing quantity with median:2.0
filled missing category with 'Unknown category'
total after missing fix:0


In [23]:
print(f"rows before deduplication:{len(ai)}")
print(f"Duplicate rows found: {ai.duplicated().sum()}")
duplicate_mask=ai.duplicated(keep=False)
print("\nThe duplicate row(all copies shown):")
print(ai[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))
ai.drop_duplicates(inplace=True)
ai.reset_index(drop=True,inplace=True)
print(f"\nRows AFTER deduplications:{len(ai)}")
print(f"Rows dropped:{len(raw_df)-len(ai)}")


rows before deduplication:30
Duplicate rows found: 0

The duplicate row(all copies shown):
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows AFTER deduplications:30
Rows dropped:0


In [24]:
print("Sample data Before parsing:")
print(df['order_date'].head(10).tolist())
df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')
nat_count=df['order_date'].isnull().sum()
print(f"\n Unparseable dates remaining: {nat_count}")
df['year'] =df['order_date'].dt.year
df['month_name'] =df['order_date'].dt.strftime('%B')
df['day_name'] =df['order_date'].dt.strftime('%A')
print("\n Sample date After parsing:")
print(df[['order_date','year','month_name','day_name']].head(10).to_string())

Sample data Before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']

 Unparseable dates remaining: 0

 Sample date After parsing:
  order_date  year month_name   day_name
0 2024-01-05  2024    January     Friday
1 2024-01-07  2024    January     Sunday
2 2024-01-08  2024    January     Monday
3 2024-01-10  2024    January  Wednesday
4 2024-01-05  2024    January     Friday
5 2024-01-07  2024    January     Sunday
6 2024-01-12  2024    January     Friday
7 2024-01-13  2024    January   Saturday
8 2024-01-15  2024    January     Monday
9 2024-01-15  2024    January     Monday


In [25]:
print("Names Before Standardization(sample):")
print(df['customer_name'].unique()[:8].tolist())

df['customer_name']=df['customer_name'].str.strip().str.title()
print("\nNames After Standardization(sample):")
print(df['customer_name'].unique()[:8].tolist())

Names Before Standardization(sample):
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', nan, 'Ananya Das']

Names After Standardization(sample):
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', nan, 'Ananya Das']


In [28]:
wrong_mask=(df['product']=='Keyboard') & (df['category']=='Electronics')
print(f"rows to fix:{wrong_mask.sum()}")
print("before fix:")
print(df[wrong_mask][['order_id','product','category']])
df.loc[wrong_mask,'category']='Computers'
print("\nafter fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
df.loc[wrong_mask,'category']='Accessories'
print("\nAfter fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
# Ensure all categories are strings before sorting to avoid TypeError
print("\nAll unique categories now:",sorted(df['category'].fillna('Unknown Category').unique().tolist()))

rows to fix:0
before fix:
Empty DataFrame
Columns: [order_id, product, category]
Index: []

after fix:
Empty DataFrame
Columns: [order_id, product, category]
Index: []

After fix:
Empty DataFrame
Columns: [order_id, product, category]
Index: []

All unique categories now: ['Accessories', 'Electronics', 'Unknown Category']


In [30]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').fillna(0).astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print("revenue column created successfully!")
print("\nsample:quantity x unit_price=revenue")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))

revenue column created successfully!

sample:quantity x unit_price=revenue
customer_name    product  quantity  unit_price  revenue
 Ramesh Kumar     Laptop         2       45000    90000
   Priya Nair        NaN         1       15000    15000
   Amit Verma   Keyboard         3        1200     3600
 Sunita Patel    Monitor         0       22000        0
 Ramesh Kumar     Laptop         2       45000    90000
  Kiran Mehta      Mouse        10         800     8000
 Deepak Singh Headphones         2        3500     7000
          NaN     Webcam         1        2500     2500


In [31]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")

  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 4
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 13
  DATA QUALITY SCORE : 80/100
  DATA IS CLEAN      : False

  ACTION REQUIRED: Missing values detected.
  → Use df['column'].fillna(value, inplace=True)
  → For numbers: fillna(df['column'].median())
  → For text   : fillna('Unknown')


In [32]:
output_filename='clean_sales_data.csv'
df.to_csv(output_filename,index=False)
print(f"Cleaned data saved to {output_filename}")
print(f"Final dataset:{df.shape[0]} rows x {df.shape[1]} columns")
print("\nETL pipeline for sales data:COMPLETE")
print("EXTRACT ->messy_sales_data.csv")
print("TRANSFORM ->clean_sales_data.csv")
print("LOAD ->clean_sales_data.csv saved")

Cleaned data saved to clean_sales_data.csv
Final dataset:30 rows x 13 columns

ETL pipeline for sales data:COMPLETE
EXTRACT ->messy_sales_data.csv
TRANSFORM ->clean_sales_data.csv
LOAD ->clean_sales_data.csv saved


In [33]:
serp_api_key='e3fbe4dc1a0be0fa382f057c9e8806aa4058c66db4e4415df93a00f974db40a7'
serp_url='https://serpapi.com/search.json'
search_query='data engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if serp_api_key != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"search query : {search_query}")



SerpAPI Key  : Set (live data)
search query : data engineer India


In [34]:
# ============================================================
# CELL 22 — EXTRACT: Fetch Job Listings from SerpAPI
# ============================================================


def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(serp_url, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if serp_api_key != 'YOUR_SERPAPI_KEY_HERE':
    print(f"Fetching job listings for: '{search_query}'")
    job_records = fetch_jobs(search_query, serp_api_key)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")





Fetching job listings for: 'data engineer India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [35]:
import pandas as pd
df_jobs = pd.DataFrame(job_records)
csv_filename = "data_engineer_jobs_india.csv"

df_jobs.to_csv(csv_filename, index=False, encoding='utf-8')

print(f"CSV file saved successfully: {csv_filename}")
print(df_jobs.head())

CSV file saved successfully: data_engineer_jobs_india.csv
                                        title  \
0  Full Stack Data Engineer - Methods & Tools   
1       Senior Data Engineer (India - Remote)   
2                       Data Engineer_7+Years   
3                 Data Engineer with Power BI   
4           Data Engineer III, RAG and Gen AI   

                             company                        location  \
0  1127 Airbus India Private Limited                           India   
1                           Linkedin  Varanasi, Uttar Pradesh, India   
2                           Zorba AI     Bengaluru, Karnataka, India   
3               Celebal Technologies        Pune, Maharashtra, India   
4                      Expedia Group        Gurugram, Haryana, India   

         posted              salary   job_type  \
0  15 hours ago       Not Disclosed  Full-time   
1   9 hours ago       Not Disclosed  Full-time   
2    2 days ago  ₹1.5M–₹2.2M a year  Full-time   
3    3 days ag

In [36]:

output_filename = 'clean_data_engineer_jobs_india.csv'
df_jobs.to_csv(output_filename, index=False, encoding='utf-8')
print(f"Cleaned data saved to {output_filename}")
print(f"Final dataset: {df_jobs.shape[0]} rows x {df_jobs.shape[1]} columns")

print("\nETL pipeline for job data: COMPLETE")
print("EXTRACT -> data_engineer_jobs_india.csv")
print("TRANSFORM -> clean_data_engineer_jobs_india.csv")
print("LOAD -> clean_data_engineer_jobs_india.csv saved")

Cleaned data saved to clean_data_engineer_jobs_india.csv
Final dataset: 20 rows x 7 columns

ETL pipeline for job data: COMPLETE
EXTRACT -> data_engineer_jobs_india.csv
TRANSFORM -> clean_data_engineer_jobs_india.csv
LOAD -> clean_data_engineer_jobs_india.csv saved
